# 01 — Bronze & Silver

Bronze raw + lineage, idempotent by `_batch_id`. Silver typed, deduped, HMAC masked, quarantine.

In [1]:
import pathlib
# live bronze/silver counts if lake up
try:
    from jobs.common.spark import get_spark
    spark = get_spark("nb01")
    for layer in ["bronze","silver"]:
        for tbl in ["customers","accounts","transactions"]:
            t = f"banking.{layer}.{tbl}"
            try:
                print(t, spark.table(t).count())
            except Exception as e:
                print(t, "missing", str(e)[:60])
    spark.stop()
except Exception as e:
    print("lake not up:", str(e)[:100])
# DDL
print(open(pathlib.Path("sql/bronze_ddl.sql") if pathlib.Path("sql/bronze_ddl.sql").exists() else pathlib.Path("../sql/bronze_ddl.sql")).read().splitlines()[0:3])


lake not up: No module named 'jobs'
['-- sql/bronze_ddl.sql â€” Bronze Iceberg DDL per Architecture 5.3 + 6.3/6.4 (Phase 1)', '-- Catalog: banking (JdbcCatalog on postgres banking_dw), warehouse s3://banking-lakehouse/local/warehouse', '-- Run via Spark session factory (jobs/common/spark.py) or spark-sql: spark.sql("CREATE TABLE banking.bronze.<table> ...")']


In [2]:
import pathlib
# dedup logic
# Window partitionBy business_key orderBy _source_ts DESC, _id DESC then rn=1
print(open(pathlib.Path("jobs/transform/silver_customers.py") if pathlib.Path("jobs/transform/silver_customers.py").exists() else pathlib.Path("../jobs/transform/silver_customers.py")).read()[400:900])


unctions as F
from pyspark.sql.types import (
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

from jobs.common.logging import get_logger
from jobs.common.spark import get_spark

logger = get_logger("silver_customers")

CUSTOMER_SCHEMA = StructType(
    [
        StructField("customer_id", IntegerType(), False),
        StructField("name", StringType(), True),
        StructField("gender", StringType(), True),
        StructField("date_of_birth", StringType


In [3]:
# HMAC masked PII - raw never reaches serving
import hmac, hashlib, os
s = os.getenv("PII_HMAC_SECRET","demo_secret"); sl = os.getenv("PII_HMAC_SALT","demo_salt")
def h(v): return hmac.new((s+sl).encode(), v.encode(), hashlib.sha256).hexdigest()
print(h("alice@example.com")[:8]+"***")
# Silver UDF does this for email/phone


d9825db5***


In [4]:
# quarantine
try:
    from jobs.common.spark import get_spark
    spark = get_spark("nb01-q")
    for tbl in ["customers","transactions"]:
        try:
            print("quarantine", tbl, spark.table(f"banking.quarantine.{tbl}").count())
        except:
            print("quarantine", tbl, 0)
    spark.stop()
except Exception as e:
    print("skip", str(e)[:60])


skip No module named 'jobs'
